# Urban Traffic Object Detection with YOLOv5

Academic group project, 2024. This notebook is an archived Colab workflow. See the repository README for setup, results, attribution and known limitations.


# 1.Objetivo

Implementar un sistema de deteccion de objetos capaz de identificar y localizar vehiculos, peatones y señales de trafico en imagenes de escenas urbanas.

# Enunciado

Utilizando un modelo preentrenado como YOLO o Faster R-CNN, desarrolle un script en Python en Google Colab, que permita:

1.   Cargar imágenes de escenas urbanas (cargado desde una dirección o carpeta).
2.   Detectar y localizar vehículos, peatones y señales de tráfico en estas imágenes.
3.   Dibujar bounding boxes alrededor de los objetos detectados y etiquetar cada objeto.
4.   Calcular y mostrar la confianza de cada detección.
5.   Generar un resumen con el conteo de cada tipo de objeto detectado en la imagen.

# 2.Dataset

Este dataset se elaboro con una mezcla de diversos dataset encontrados en roboflow (Links de estos en los anexos) de vision computacional, especificamente de escenas urbanas, y datasets destinados a automoviles autonomos

Las etiquetas de interes para este dataset son:



1.   Car(Automovil)
2.   Bus
3.   Motorcycle(Motocicleta)
4.   Truck(Camioneta)
5.   Person(Persona)
6.   Signal(Señaletica)
7.   Bicycle(Bicicleta)
8.   Ambulance(Ambulancia)
9.   Traffic light(Semaforo)

#3.Pre Procesamiento

Para ejecutar este notebook de forma standalone, se compartira el drive respectivo donde esta almacenado el dataset utilizado, por lo que se montara el drive y se descomprimira el dataset respectivo en la carpeta /content/Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/Shareddrives/Inteligencia\ artificial-entrega\ de\ trabajos/Datasets/Dataset_AcP4 -d /content/Dataset

Se utiliza la libreria albumentations para realizar Data Augmentation al modelo

In [ ]:
!pip install albumentations

In [ ]:
!pip install pyyaml

Para este trabajo se utilizara YOLOv5 por su amplitud de documentacion disponible

In [ ]:
# Clonar el repositorio YOLOv5
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5

# Instalar las dependencias
!pip install -r requirements.txt

Carga respectiva de todas las librerias utilizadas

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np
from google.colab import files
from IPython.display import Image, display
import albumentations as A
import shutil
import pandas as pd
from PIL import Image as PILImage
import matplotlib.image as mpimg
import yaml

Se definen los directorios en donde se almacenaran las imagenes y etiquetas para su split respectivo: Train, Valid y Test

In [ ]:
directories = {
    'train_images': '/content/Dataset/train/images/',
    'train_labels': '/content/Dataset/train/labels/',
    'valid_images': '/content/Dataset/valid/images/',
    'valid_labels': '/content/Dataset/valid/labels/',
    'test_images': '/content/Dataset/test/images/',
    'test_labels': '/content/Dataset/test/labels/'
}

Se carga el .yaml del dataset para extraer el nombre de las etiquetas, verificando despues que se realice correctamente este proceso

In [ ]:
# Ruta a tu archivo data.yaml
data_yaml_path = '/content/Dataset/data.yaml'

# Cargar el archivo YAML
with open(data_yaml_path, 'r') as file:
    data = yaml.safe_load(file)

# Extraer los nombres de las clases
class_names = data['names']

# Verificar que se han cargado las clases correctamente
print(f"Nombres de las clases: {class_names}")


Funcion para dibujar los bounding boxes sobre la imagen
---



```
image = cv2.imread(image_path)
img_height, img_width, _ = image.shape

# Leer las etiquetas en formato YOLO
with open(label_path, 'r') as file:
    lines = file.readlines()
```

# Se define la variable donde se almacenara la imagen en image_path junto con su tamaño como image.shape



```
for line in lines:
    class_id, x_center, y_center, width, height = map(float, line.strip().split())
    x_min = int((x_center - width / 2) * img_width)
    y_min = int((y_center - height / 2) * img_height)
    x_max = int((x_center + width / 2) * img_width)
    y_max = int((y_center + height / 2) * img_height)
```


La función draw_bounding_boxes carga una imagen y dibuja cuadros delimitadores (bounding boxes) basados en etiquetas YOLO. Primero, lee las dimensiones de la imagen y luego extrae las coordenadas normalizadas del archivo de etiquetas. Estas coordenadas se convierten a valores absolutos en píxeles para obtener los bordes del cuadro delimitador. Luego, utiliza cv2.rectangle para dibujar los cuadros en la imagen y cv2.putText para mostrar el nombre de la clase del objeto detectado. Finalmente, devuelve la imagen con los cuadros y etiquetas.

In [ ]:
# Función para dibujar los bounding boxes sobre la imagen
def draw_bounding_boxes(image_path, label_path, class_names):
    image = cv2.imread(image_path)
    img_height, img_width, _ = image.shape

    # Leer las etiquetas en formato YOLO
    with open(label_path, 'r') as file:
        lines = file.readlines()

    # Dibujar cada cuadro delimitador
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        x_min = int((x_center - width / 2) * img_width)
        y_min = int((y_center - height / 2) * img_height)
        x_max = int((x_center + width / 2) * img_width)
        y_max = int((y_center + height / 2) * img_height)

        # Dibujar el cuadro delimitador en verde
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

        # Dibujar la etiqueta de clase
        label = f"{class_names[int(class_id)]}"
        cv2.putText(image, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    return image


Esta función, get_image_sizes, obtiene las dimensiones (ancho y alto) de todas las imágenes en un directorio específico.



*   1) Inicializa una lista vacía image_sizes para almacenar los tamaños de las imágenes.

*   2) Itera sobre todos los archivos en el directorio dado.

*   3) Si el archivo tiene una extensión de imagen común (como .jpg, .png, o .jpeg), se abre la imagen usando PILImage.

*   4) Obtiene las dimensiones de la imagen con img.size y las guarda en la lista.

*   5) Finalmente, devuelve la lista con los tamaños de todas las imágenes del directorio.

In [ ]:
# Función para obtener las dimensiones de las imágenes
def get_image_sizes(directory):
    image_sizes = []
    for filename in os.listdir(directory):
        if filename.endswith(('.jpg', '.png', '.jpeg')):  # Extensiones comunes de imágenes
            image_path = os.path.join(directory, filename)
            with PILImage.open(image_path) as img:  # Usar PILImage aquí
                image_sizes.append(img.size)  # Guardar tamaño (ancho, alto)
    return image_sizes

Esta función, resize_images_in_directory, redimensiona todas las imágenes en un directorio a un tamaño de 640x640 píxeles (por defecto).

*   1) Lista los archivos de imagen (con extensiones .jpg o .png) en el directorio dado.

*   2) Para cada archivo de imagen, lo carga utilizando cv2.imread.

*   3) Si la imagen se carga correctamente, la redimensiona usando cv2.resize con el tamaño especificado (640x640).

*   4) Guarda la imagen redimensionada en su ubicación original utilizando cv2.imwrite, reemplazando el archivo original.

*   5) Si la imagen no puede leerse, imprime un mensaje de error.


In [ ]:
# Función para redimensionar imágenes a 256x256
def resize_images_in_directory(directory, size=(640, 640)):
    # Listar todas las imágenes en el directorio
    image_files = [f for f in os.listdir(directory) if f.endswith('.jpg') or f.endswith('.png')]

    for image_file in image_files:
        image_path = os.path.join(directory, image_file)
        image = cv2.imread(image_path)

        if image is not None:
            # Redimensionar la imagen
            resized_image = cv2.resize(image, size)

            # Guardar la imagen redimensionada (reemplazando la original)
            cv2.imwrite(image_path, resized_image)
            print(f"Imagen {image_file} redimensionada y guardada en {image_path}")
        else:
            print(f"No se pudo leer la imagen {image_file}")


Estas dos funciones trabajan juntas para obtener y verificar las dimensiones de las imágenes en un conjunto de directorios.

I) get_image_sizes:



*   1) Recorre un directorio específico,
cargando imágenes (extensiones .jpg o .png) usando OpenCV (cv2.imread).

*   2) Obtiene las dimensiones (ancho y alto) de cada imagen y las guarda junto con el nombre del archivo en una lista.

*   3) Devuelve esta lista de tamaños de imagen.


II) check_largest_and_smallest_image_sizes:



*   Recorre múltiples directorios de imágenes, llamando a get_image_sizes para obtener los tamaños.

*   Encuentra la imagen más grande y la más pequeña (basado en el área: ancho * alto).

*   Imprime las dimensiones y nombres de las imágenes más grande y más pequeña.

In [ ]:
# Función para obtener las dimensiones de las imágenes en un directorio
def get_image_sizes(directory):
    image_sizes = []
    image_files = [f for f in os.listdir(directory) if f.endswith('.jpg') or f.endswith('.png')]

    for image_file in image_files:
        image_path = os.path.join(directory, image_file)
        image = cv2.imread(image_path)

        if image is not None:
            height, width, _ = image.shape
            image_sizes.append((width, height, image_file))
        else:
            print(f"No se pudo leer la imagen {image_file} en {directory}")

    return image_sizes

# Función para verificar las dimensiones de todas las imágenes
def check_largest_and_smallest_image_sizes(directories):
    all_image_sizes = []

    # Recorrer todos los directorios de imágenes
    for dir_key, dir_path in directories.items():
        if 'images' in dir_key:  # Solo procesar carpetas de imágenes
            all_image_sizes.extend(get_image_sizes(dir_path))

    # Encontrar las dimensiones de la imagen más grande y más pequeña
    if all_image_sizes:
        largest_image = max(all_image_sizes, key=lambda x: x[0] * x[1])  # Ancho * Alto
        smallest_image = min(all_image_sizes, key=lambda x: x[0] * x[1])

        print(f"Dimensiones de la imagen más grande: {largest_image[0]}x{largest_image[1]} píxeles ({largest_image[2]})")
        print(f"Dimensiones de la imagen más pequeña: {smallest_image[0]}x{smallest_image[1]} píxeles ({smallest_image[2]})")
    else:
        print("No se encontraron imágenes.")


Este código visualiza las primeras 10 imágenes del conjunto de entrenamiento con sus respectivas cajas delimitadoras (bounding boxes).



*   1) Directorio de imágenes y etiquetas: Se obtienen las rutas de los directorios de imágenes y etiquetas de entrenamiento.

*   2) Seleccionar archivos: Se seleccionan las primeras 10 imágenes y sus etiquetas correspondientes.

*   3) Crear subplots: Se configura una cuadrícula de gráficos de 5 filas y 2 columnas para mostrar las 10 imágenes.

*   4) Dibujar bounding boxes: Para cada imagen, se llama a la función draw_bounding_boxes para dibujar los cuadros delimitadores sobre la imagen.

*   5) Convertir formato: Las imágenes se convierten de BGR (usado por OpenCV) a RGB (usado por matplotlib).

*   6) Mostrar imágenes: Las imágenes con sus bounding boxes se muestran en la cuadrícula sin ejes, junto con el título de cada imagen (su nombre).

In [ ]:
# Visualizar las primeras 10 imágenes del conjunto de entrenamiento
images_dir = directories['train_images']
labels_dir = directories['train_labels']

image_files = sorted(os.listdir(images_dir))[:10]
label_files = sorted(os.listdir(labels_dir))[:10]

# Graficar las imágenes con las cajas delimitadoras (5 filas x 2 columnas)
fig, axs = plt.subplots(5, 2, figsize=(20, 50))
axs = axs.ravel()

for i, (image_file, label_file) in enumerate(zip(image_files, label_files)):
    image_path = os.path.join(images_dir, image_file)
    label_path = os.path.join(labels_dir, label_file)

    # Dibujar las cajas delimitadoras en la imagen
    image_with_boxes = draw_bounding_boxes(image_path, label_path, class_names)

    # Convertir de BGR a RGB para matplotlib
    image_with_boxes_rgb = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)

    # Mostrar la imagen
    axs[i].imshow(image_with_boxes_rgb)
    axs[i].axis('off')
    axs[i].set_title(image_file)

plt.tight_layout()
plt.show()

In [ ]:
check_largest_and_smallest_image_sizes(directories)


Estas dos funciones permiten leer y escribir etiquetas en formato YOLO, que es comúnmente utilizado en tareas de detección de objetos.

I) read_yolo_labels:



*   1) Abre un archivo de etiquetas en formato YOLO, donde cada línea contiene la clase del objeto y las coordenadas normalizadas del bounding box (centro, ancho, altura).

*   2) Extrae las coordenadas de los bounding boxes y las etiquetas de clase, almacenándolas en listas separadas (boxes y class_labels).

*   3) Devuelve estas listas.


II) write_yolo_labels:

*   1) Escribe etiquetas en formato YOLO en un archivo.

*   2) Para cada bounding box y su clase asociada, escribe una línea con el formato class_id x_center y_center width height, donde las coordenadas están normalizadas.

In [ ]:
# Función para leer los labels en formato YOLO
def read_yolo_labels(label_path):
    with open(label_path, 'r') as file:
        lines = file.readlines()
    boxes = []
    class_labels = []
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        boxes.append([x_center, y_center, width, height])  # Solo las coordenadas, sin la clase
        class_labels.append(int(class_id))  # Solo la clase
    return boxes, class_labels

# Función para escribir los labels en formato YOLO
def write_yolo_labels(label_path, boxes, class_labels):
    with open(label_path, 'w') as file:
        for box, class_id in zip(boxes, class_labels):
            x_center, y_center, width, height = box
            file.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

Este código define un conjunto de transformaciones de Data Augmentation utilizando la librería albumentations. Las transformaciones ayudan a aumentar la variabilidad del conjunto de datos y mejorar la robustez del modelo de detección de objetos.



*   1) RandomRotate90: Realiza una rotación aleatoria de 90 grados con una probabilidad del 50%.

*   2) ShiftScaleRotate: Aplica un desplazamiento, escalado o rotación aleatorios dentro de los límites dados (rotación entre -45 y 45 grados) con una probabilidad del 70%.

*   3) Blur: Aplica un desenfoque aleatorio ligero con una probabilidad del 20%.

*   4) HorizontalFlip: Voltea la imagen horizontalmente con una probabilidad del 50%.

*   5) bbox_params: Define que las cajas delimitadoras están en formato YOLO y se deben ajustar según las transformaciones aplicadas.








In [ ]:
# Transformaciones de Data Augmentation
transform = A.Compose([
    A.RandomRotate90(p=0.5),  # Rotación aleatoria de 90 grados
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.7),  # Rotaciones aleatorias entre -45 y 45 grados
    A.Blur(blur_limit=3, p=0.2),  # Aplicar un desenfoque ligero aleatorio
    A.HorizontalFlip(p=0.5),  # Volteo horizontal
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

Este código prepara el entorno para aplicar Data Augmentation a las imágenes de entrenamiento y almacenar los resultados en nuevas carpetas.



*   1) Crear directorios:
  *   Crea dos nuevas carpetas: una para almacenar las imágenes aumentadas y otra para sus correspondientes etiquetas.
  *   Se asegura de que los directorios existen utilizando os.makedirs con exist_ok=True.

*   2) Definir directorios de imágenes:
  *   Se asegura de que los directorios existen utilizando os.makedirs con exist_ok=True.


*   3) Obtener lista de archivos de imágenes:
  *   Filtra los archivos en el directorio de imágenes de entrenamiento, seleccionando solo aquellos con extensión .jpg o .png.
    

*   4) Definir cantidad de augmentations:
  *   Establece que por cada imagen original se generarán 3 imágenes aumentadas, para aumentar la variabilidad del conjunto de datos.


In [ ]:
# Crear una nueva carpeta para almacenar las imágenes aumentadas
augmented_images_dir = '/content/train_augmented/images'
augmented_labels_dir = '/content/train_augmented/labels'

os.makedirs(augmented_images_dir, exist_ok=True)
os.makedirs(augmented_labels_dir, exist_ok=True)

# Aplicar Data Augmentation solo a las imágenes de entrenamiento
images_dir = directories['train_images']
labels_dir = directories['train_labels']

# Lista de archivos de imágenes
image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg') or f.endswith('.png')]

# Definir cuántas imágenes aumentadas deseas generar por imagen original
num_augmented_images_per_original = 3  # Generar 3 variaciones de cada imagen original

Este código aplica Data Augmentation a un conjunto de imágenes y sus etiquetas en formato YOLO, generando múltiples variaciones de cada imagen, y luego guarda tanto las imágenes aumentadas como sus etiquetas correspondientes.



*   1) Recorrer imágenes:
  *   Para cada imagen en el directorio de entrenamiento, se obtiene la ruta de la imagen y de su archivo de etiquetas.

*   2) Cargar imagen y etiquetas:
  *   Carga la imagen y sus dimensiones.
  *   Lee las etiquetas en formato YOLO (bounding boxes y clases) utilizando la función read_yolo_labels.

*   3) Aplicar Data Augmentation:
  *   Genera varias imágenes aumentadas (definido por num_augmented_images_per_original).
  *   Aplica las transformaciones definidas previamente a la imagen y sus bounding boxes.
  *   Guarda las imágenes aumentadas y sus etiquetas asociadas en nuevas carpetas con un sufijo (aug_x_).

*   4) Copiar imágenes originales:
  *   Copia las imágenes y etiquetas originales sin modificar a las carpetas de imágenes aumentadas.

*   5) Mensaje final:
  *   Imprime un mensaje indicando que el proceso de augmentación ha sido completado.

In [ ]:
# Aplicar Data Augmentation
for image_file in image_files:
    image_path = os.path.join(images_dir, image_file)
    label_file = image_file.replace('.jpg', '.txt').replace('.png', '.txt')
    label_path = os.path.join(labels_dir, label_file)

    # Cargar imagen y obtener sus dimensiones
    image = cv2.imread(image_path)
    height, width, _ = image.shape

    # Leer las etiquetas (coordenadas + clases)
    boxes, class_labels = read_yolo_labels(label_path)

    # Generar múltiples imágenes aumentadas
    for i in range(num_augmented_images_per_original):
        # Aplicar la transformación
        augmented = transform(image=image, bboxes=boxes, class_labels=class_labels)
        augmented_image = augmented['image']
        augmented_bboxes = augmented['bboxes']
        augmented_class_labels = augmented['class_labels']

        # Guardar la imagen aumentada con un sufijo para evitar sobrescribir
        augmented_image_path = os.path.join(augmented_images_dir, f"aug_{i}_{image_file}")
        cv2.imwrite(augmented_image_path, augmented_image)

        # Guardar las etiquetas aumentadas con el mismo sufijo
        augmented_label_path = os.path.join(augmented_labels_dir, f"aug_{i}_{label_file}")
        write_yolo_labels(augmented_label_path, augmented_bboxes, augmented_class_labels)

        print(f"Imagen aumentada {i} para {image_file} guardada.")

# Copiar las imágenes originales sin modificar a la nueva carpeta (opcional)
for image_file in image_files:
    original_image_path = os.path.join(images_dir, image_file)
    original_label_file = image_file.replace('.jpg', '.txt').replace('.png', '.txt')
    original_label_path = os.path.join(labels_dir, original_label_file)

    shutil.copy(original_image_path, augmented_images_dir)
    shutil.copy(original_label_path, augmented_labels_dir)

print("Proceso de augmentación completado.")


Este código recorre directorios de imágenes, obtiene sus dimensiones y encuentra la imagen más grande y más pequeña.

*   1) Recorrido de directorios:
  *   Se recorre cada directorio en el diccionario directories. Solo se procesan aquellos directorios cuyo nombre contiene la palabra 'images'.
  *   Para cada directorio de imágenes, llama a la función get_image_sizes para obtener las dimensiones de todas las imágenes y las almacena en all_image_sizes.

*   2) Encontrar la imagen más grande y más pequeña:
  *   Utiliza la función max para encontrar la imagen con el área más grande (ancho * alto).
  *   Utiliza la función min para encontrar la imagen con el área más pequeña.

*   3) Impresión de resultados:
  *   Si se encontraron imágenes, imprime las dimensiones de la imagen más grande y más pequeña.
  *   Si no se encontraron imágenes, imprime un mensaje indicando que no se encontraron.

In [ ]:
# Recorrer todos los directorios de imágenes
all_image_sizes = []
for dir_key, dir_path in directories.items():
    if 'images' in dir_key:  # Solo procesar carpetas de imágenes
        all_image_sizes.extend(get_image_sizes(dir_path))

# Encontrar las dimensiones de la imagen más grande y más pequeña
if all_image_sizes:
    largest_image = max(all_image_sizes, key=lambda x: x[0] * x[1])  # Ancho * Alto
    smallest_image = min(all_image_sizes, key=lambda x: x[0] * x[1])

    print(f"Dimensiones de la imagen más grande: {largest_image}")
    print(f"Dimensiones de la imagen más pequeña: {smallest_image}")
else:
    print("No se encontraron imágenes.")

Este código cuenta e imprime la cantidad de imágenes en el directorio de imágenes de entrenamiento.

*   1) Definir directorio:
  *   Establece la ruta al directorio donde están almacenadas las imágenes aumentadas de entrenamiento (train_images_dir).

*   2) Contar archivos de imagen:
  *   Filtra los archivos en el directorio que tienen las extensiones .jpg o .png y los almacena en una lista (image_files).

*   3) Imprimir la cantidad:
  *   Imprime la cantidad total de imágenes en el directorio, utilizando la longitud de la lista image_files.

In [ ]:
# Directorio de imágenes de entrenamiento
train_images_dir = '/content/train_augmented/images/'

# Contar archivos de imagen en la carpeta
image_files = [f for f in os.listdir(train_images_dir) if f.endswith('.jpg') or f.endswith('.png')]

# Imprimir la cantidad de imágenes
print(f"Cantidad de imágenes en {train_images_dir}: {len(image_files)}")


Este comando entrena un modelo de detección de objetos utilizando YOLOv5 con una serie de parámetros específicos:

*   1) train.py: Ejecuta el script de entrenamiento de YOLOv5.

*   2) --img 512: Establece el tamaño de las imágenes de entrada a 512x512 píxeles.

*   3) --batch 32: Define el tamaño del lote en 32 imágenes por iteración.

*   4) --epochs 20: Entrena el modelo durante 20 épocas.

*   5) --data /content/data.yaml: Especifica la ruta al archivo de configuración data.yaml, que define el conjunto de datos.

*   6) --weights yolov5s.pt: Usa los pesos preentrenados del modelo YOLOv5 pequeño (yolov5s.pt) como punto de partida.

*   7) --cache: Carga las imágenes en la memoria caché para acelerar el proceso de entrenamiento

In [ ]:
!python train.py --img 512 --batch 32 --epochs 20 --data /content/data.yaml --weights yolov5s.pt --cache


Este comando utiliza la función files.download de Google Colab para descargar un archivo desde el entorno de ejecución a tu máquina local. En este caso, el archivo descargado es el modelo entrenado (best.pt), que se encuentra en la ruta /content/yolov5/runs/train/exp/weights/best.pt.

El archivo best.pt es el modelo YOLOv5 entrenado que ha obtenido los mejores resultados durante el proceso de entrenamiento, y este comando te permite guardarlo localmente.

In [ ]:
files.download('/content/yolov5/runs/train/exp/weights/best.pt')

Este código carga los resultados del entrenamiento de un modelo YOLOv5 desde un archivo CSV y genera gráficos para visualizar las métricas clave.

*   1) Cargar el archivo CSV:
  *   Se especifica la ruta del archivo CSV que contiene los resultados del entrenamiento (results_csv_path).
  *   Se cargan los datos en un DataFrame de pandas, eliminando los posibles espacios extra en los nombres de las columnas.

*   2) Verificación de las columnas:
  *   Imprime los nombres de las columnas para asegurarse de que no haya errores o espacios no deseados.

*   3) Gráfico de pérdidas (losses):
  *   Grafica las pérdidas durante el entrenamiento y validación para las cajas (box), objetos (object) y clases (class).
  *   Muestra cómo evolucionan las pérdidas a lo largo de las épocas.

*   4) Gráfico de precisión, recall y mAP:
  *   Grafica las métricas de precisión, recall, y mAP (media de la precisión promedio), que son indicadores clave del rendimiento del modelo en las tareas de detección de objetos.
  *   Visualiza cómo estas métricas cambian a lo largo del entrenamiento.

In [ ]:
# Ruta al archivo CSV con los resultados del entrenamiento
results_csv_path = '/content/yolov5/runs/train/exp/results.csv'  # Ajusta según sea necesario

# Cargar los datos del CSV y eliminar espacios extra en los nombres de las columnas
df = pd.read_csv(results_csv_path)
df.columns = df.columns.str.strip()  # Eliminar espacios extra en los nombres de las columnas

# Verifica que los nombres de las columnas ahora estén bien
print(df.columns)

# Gráfico de las pérdidas (losses)
plt.figure(figsize=(10, 5))
plt.plot(df['train/box_loss'], label='Train Box Loss')
plt.plot(df['train/obj_loss'], label='Train Object Loss')
plt.plot(df['train/cls_loss'], label='Train Class Loss')
plt.plot(df['val/box_loss'], label='Validation Box Loss')
plt.plot(df['val/obj_loss'], label='Validation Object Loss')
plt.plot(df['val/cls_loss'], label='Validation Class Loss')
plt.title('Pérdida durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

# Gráfico de precisión, recall, y mAP
plt.figure(figsize=(10, 5))
plt.plot(df['metrics/precision'], label='Precision')
plt.plot(df['metrics/recall'], label='Recall')
plt.plot(df['metrics/mAP_0.5'], label='mAP@0.5')
plt.plot(df['metrics/mAP_0.5:0.95'], label='mAP@0.5:0.95')
plt.title('Precisión, Recall y mAP durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()


Este código carga y muestra una imagen de la matriz de confusión generada durante el entrenamiento del modelo YOLOv5.



*   1) Ruta de la imagen:
  *   Define la ruta de la imagen de la matriz de confusión (confusion_matrix.png) en el directorio donde se almacenan los resultados del entrenamiento.

*   2) Cargar la imagen:
  *   Configura una figura de tamaño 12x12 con plt.figure.

*   3) Mostrar la imagen:
  *   Utiliza mpimg.imread para cargar la imagen en formato PNG.
  *   Muestra la imagen cargada con plt.imshow y oculta los ejes con plt.axis('off') para centrarse solo en la visualización de la matriz.


  Esto es útil para analizar el desempeño del modelo en la clasificación de las clases de objetos detectados.



In [ ]:
# Cargar y mostrar la imagen de la matriz de confusión
img_path = '/content/yolov5/runs/train/exp/confusion_matrix.png'  # Ajusta esta ruta según corresponda

# Cargar la imagen
img = mpimg.imread(img_path)

# Mostrar la imagen
plt.figure(figsize=(12, 12))  # Ajusta el tamaño de la figura
plt.imshow(img)
plt.axis('off')  # Ocultar los ejes
plt.show()


Este código permite cargar una imagen, procesarla con un modelo YOLOv5 entrenado y visualizar los resultados con los cuadros delimitadores de los objetos detectados.



*   1) Subir imagen:
  *   Utiliza files.upload() para permitir al usuario subir una imagen desde su dispositivo.
  *   Obtiene el nombre del archivo subido y lo imprime.

*   2) Renombrar archivo:
  *   Si el nombre de la imagen contiene espacios o paréntesis, el archivo se renombra para eliminar estos caracteres y evitar problemas de ruta.
  *   Usa os.rename para aplicar los cambios.

*   3) Ejecutar YOLOv5:
  *   Ejecuta el script detect.py para realizar la detección de objetos en la imagen subida, utilizando el modelo entrenado (best.pt).
  *   Especifica un tamaño de imagen de 512 píxeles, un umbral de confianza de 0.15, y utiliza la opción --exist-ok para permitir sobrescribir resultados existentes en el directorio de salida.

*   4) Visualizar los resultados:
  *   Itera sobre el directorio de salida (/content/yolov5/runs/detect/exp/) para mostrar las imágenes que contienen los bounding boxes de los objetos detectados.



In [ ]:
# Subir imagen
uploaded = files.upload()

# Obtener el nombre de la imagen subida
image_path = list(uploaded.keys())[0]
print(f"Imagen cargada: {image_path}")

# Renombrar el archivo si contiene paréntesis o espacios
new_image_path = image_path.replace(' ', '_').replace('(', '').replace(')', '')
os.rename(image_path, new_image_path)
print(f"Archivo renombrado a: {new_image_path}")

# 2. Ejecutar YOLOv5 con el modelo entrenado en la imagen subida y usar la opción --exist-ok
!python detect.py --weights runs/train/exp/weights/best.pt --img 512 --conf 0.15 --source {new_image_path} --exist-ok

# 3. Visualizar los resultados
output_dir = '/content/yolov5/runs/detect/exp/'

# Mostrar las imágenes con los bounding boxes detectados
for img_file in os.listdir(output_dir):
    if img_file.endswith(('.png', '.jpg', '.jpeg')):  # Verificar que sean imágenes
        display(Image(filename=os.path.join(output_dir, img_file)))